# Experimento — Gradient boosting

Placeholder do experimento. O texto abaixo é o roteiro. Não há treino neste arquivo.

Conjunto de árvores em boosting, usado como comparação para os outros seis experimentos. As frentes de rede neural (camadas, unidades, logística, tangente hiperbólica, ReLU, Adam, dropout) não se aplicam aqui.


## Papel de referência

Os outros notebooks medem a distância até a configuração aceita deste aqui, com a mesma métrica e as mesmas dobras. Enquanto um bloco deste experimento ainda não foi aceito, os outros usam o último bloco já aceito. Não usam um número provisório.


## Como o experimento avança

O experimento é uma sequência de blocos. Cada bloco faz uma intervenção só: um hiperparâmetro, uma escolha de arquitetura ou uma mudança nos dados. O bloco seguinte parte da configuração que venceu, e só fica no lugar dela se a métrica de validação subir.

A métrica é escolhida uma vez, antes do bloco 0 de qualquer modelo, e vale para os sete notebooks. Cerca de 26,5% dos clientes cancelam, então a métrica tem de enxergar a classe de churn. O conjunto de teste não entra nessa escolha. Ele só é usado no bloco final, com a configuração já congelada.

Empate fica com a configuração mais simples. Cada bloco aceito registra o que mudou, a configuração completa e a média e a dispersão nas cinco dobras.

O gradient boosting deste repositório é a referência de comparação. Cada modelo anota a distância do seu melhor bloco até o boosting congelado com o mesmo protocolo. O boosting também anda por blocos; a comparação usa a versão já aceita, sem perseguir um número que ainda está mudando.


## Dados

A entrada é a saída de `if702-preprocessamento.ipynb`: as cinco dobras estratificadas e a função `preparar` para o treino final e o teste. Em cada dobra o scaler é ajustado só no treino da dobra.

O oversampling repetitivo já está feito aí. O `RandomOverSampler` só duplica o treino da dobra. A validação e o teste continuam com um cliente por linha. Nenhum bloco refaz esse balanceamento por fora das dobras.

### Estratégias de dados compartilhadas

Nenhuma, além do pré-processamento.

Quando aparecer uma estratégia nova de exploração ou de mineração, ela é escrita nesta seção e copiada para os outros seis notebooks. Vira um bloco em cada experimento, em cima da configuração que aquele modelo já tiver aceito. O bloco mexe só nos dados. O modelo fica parado.


## Frentes já definidas

A biblioteca é escolhida no bloco 0 e fica congelada. Trocar de implementação no meio da sequência quebraria a comparação. Candidatas naturais são o gradient boosting do scikit-learn, o XGBoost e o LightGBM. O bloco 0 anota qual entrou e por qual restrição de ambiente.

A taxa de aprendizagem, o número de árvores, a profundidade e o número de folhas são as frentes de complexidade e de passo. Cada uma é um bloco. A taxa anda junto com o número de árvores: taxa menor pede mais rodadas. Por isso o número de árvores não é um teto escolhido no escuro. O early stopping na validação da dobra decide quantas árvores ficam, e a paciência desse critério é o hiperparâmetro a mais (`n_iter_no_change` no scikit-learn, `early_stopping_rounds` no XGBoost e no LightGBM).

Profundidade (`max_depth`) e número de folhas (`num_leaves`, `max_leaf_nodes` ou `max_leaves`) não entram no mesmo bloco. No LightGBM a folha é o controle principal de complexidade, e a documentação de ajuste pede que `num_leaves` fique abaixo de 2 elevado à profundidade. Uma árvore que cresce por folha, sem teto de profundidade, fica funda demais para o mesmo número de folhas.

Gini ou entropia mede a impureza do corte numa árvore de classificação e numa random forest. No gradient boosting a árvore ajusta o gradiente de uma função de erro, e o corte não usa Gini nem entropia. No scikit-learn 1.9 o parâmetro `criterion` do `GradientBoostingClassifier` está obsoleto e não muda o ajuste. A frente que ocupa esse lugar no boosting é a função de erro: `log_loss` ou `exponential`. A segunda recupera o AdaBoost e só serve para duas classes. Gini e entropia ficam para um bloco lateral, se ele for aberto, comparando uma árvore única ou uma random forest com o boosting. Esse bloco só substitui o boosting se a métrica vencer.

Além dessa lista, as documentações de ajuste do LightGBM e do XGBoost colocam mais quatro famílias, cada uma num bloco:

- Mínimo de amostra na folha. `min_data_in_leaf` no LightGBM, `min_samples_leaf` no scikit-learn, `min_child_weight` no XGBoost. É o freio mais citado quando a árvore cresce por folha. Se a biblioteca expuser amostra mínima e `min_child_weight`, são dois blocos.
- Subamostra de linhas. `subsample` ou `bagging_fraction`. No LightGBM só passa a valer com `bagging_freq` maior que zero; a frequência entra no mesmo bloco, porque sem ela a fração não age.
- Subamostra de colunas. `feature_fraction`, `colsample_bytree` ou `max_features`.
- Regularização do corte e da folha. `gamma` (`min_split_loss`) é a queda mínima da função de erro para aceitar um corte. `reg_lambda` é a penalidade L2 nos pesos das folhas e `reg_alpha` é a L1. No CatBoost a L2 se chama `l2_leaf_reg`. Se a biblioteca escolhida for o CatBoost, o esquema `Ordered` ou `Plain` é um bloco próprio.

O oversampling repetitivo é o mesmo das dobras, para a comparação ver a mesma amostra que as redes. Peso de classe (`class_weight`, `scale_pos_weight`) fica desligado enquanto esse oversampling estiver ativo. Ligar os dois no mesmo bloco mistura os efeitos.


## Hiperparâmetros que ainda vão chegar

Taxa, número de árvores com early stopping, profundidade, folhas, função de erro, amostra mínima na folha, subamostra de linhas, subamostra de colunas e regularização já estão nas frentes acima. O que ainda chegar entra como bloco novo, um item de cada vez, a partir da configuração vencedora.


## Ordem prevista dos blocos

1. Bloco 0. A biblioteca escolhida, no padrão dela, com early stopping já ligado numa paciência inicial, sobre as dobras com oversampling. Esse número já pode ser citado pelos outros notebooks.
2. Profundidade.
3. Número de folhas, abaixo de 2 elevado à profundidade aceita quando a biblioteca cresce por folha.
4. Mínimo de amostra na folha. Se a biblioteca também tiver `min_child_weight`, ele vem no bloco seguinte.
5. Taxa de aprendizagem. O número de árvores é reescolhido pelo early stopping, com a paciência ainda fixa.
6. Paciência do early stopping.
7. Subamostra de linhas, e a frequência de bagging se a biblioteca exigir.
8. Subamostra de colunas.
9. Queda mínima para aceitar um corte (`gamma`), se existir.
10. Regularização L2 das folhas, depois L1, se existir. No CatBoost, o esquema `Ordered` ou `Plain` entra antes da L2.
11. Função de erro do boosting: `log_loss` e `exponential`.
12. Bloco lateral, só se for aberto: Gini e entropia numa árvore única ou numa random forest, comparadas ao boosting aceito.
13. Itens que ainda chegarem.
14. Estratégias de dados compartilhadas, na mesma ordem dos outros experimentos.
15. Bloco final no teste. A partir daqui o número de referência para o relatório é o do teste, e os outros notebooks só comparam teste com teste.


## O que este arquivo ainda não contém

Não há código, treino, busca nem número medido. Este notebook é o roteiro. O primeiro bloco só começa depois que a métrica comum estiver escrita.
